### Model Pipeline

1. Raw table_data (string)

        ↓ linearize

2. Linearized text (string)

        ↓ tokenizer.encode()

3. Token IDs (integers)

        ↓ model's embedding layer (learned lookup table, inside the model)

4. Token embeddings + positional embeddings

        ↓ encoder self-attention layers (bidirectional)

5. Encoder hidden states (contextualized representations, one vector per input token)

        ↓ fed into every decoder layer via cross-attention

6. Decoder (causal self-attention + cross-attention into step 5)

        ↓ generates one token at a time, autoregressively

7. Output logits → softmax → generated token

        ↓ repeat step 6-7 until <eos>
        
8. Decoded text = generated financial commentary

execution order:

1. Setup and preprocessing  
2. Shared datasets and evaluation utilities  
3. Zero-shot T5-small baseline  
4. Fine-tuned T5-small  
5. Combined model-level metrics and prediction-level outputs  

### Installs and Imports

In [1]:
!pip install -q  -U transformers
!pip install -q -U datasets
!pip install -q -U evaluate
!pip install -q tokenizers
!pip install rouge_score


In [22]:
!pip install rouge_score
!pip install sacrebleu
!pip install meteor-score
!pip install bert-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a150b04419923317734aff0adeec4589586671680c196e53233494ac88e98c7b
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.8 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement meteor-score (from versions: none)
ERROR: No matching distribution found for meteor-score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

In [3]:
from datasets import Dataset, DatasetDict
import evaluate

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

In [4]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [7]:
print("PyTorch version:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Using preprocessing notebook to use the same variables and features.

In [9]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project"
PREPROCESSING_SCRIPT = os.path.join(BASE_DIR, "train_splitEDA.py")

# This script must create train_df, test_df, summarize_market_table, and token_length.
%run -i "$PREPROCESSING_SCRIPT"

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (3143, 7)
Test shape: (795, 7)
['source', 'market', 'date', 'instruction', 'table_data', 'report', 'prompts']


,source,market,date,instruction,table_data,report,prompts
0,totalfarmmarketing,cattle,2021-12-02,Please act as an expert financial market analy...,Date Pro...,Cattle futures posted moderate to strong gains...,{'instruction': 'Please act as an expert finan...
1,totalfarmmarketing,cattle,2022-01-14,Please act as an expert financial market analy...,Date Pro...,"Cattle prices are trying to turn higher, as th...",{'instruction': 'Please act as an expert finan...


0
0
count    3143.000000
mean      536.570474
std       363.832138
min        12.000000
25%       271.000000
50%       451.000000
75%       690.000000
max      2243.000000
Name: report, dtype: float64
      Date                                 Product Name Symbol   Open   High    Low  Close  Volume
2021-11-02  Live Cattle Future (front month) (December)   LEZ1 128.90 130.43 128.88 129.95 21275.0
2021-11-03  Live Cattle Future (front month) (December)   LEZ1 130.90 132.40 130.62 131.65 27572.0
2021-11-04  Live Cattle Future (front month) (December)   LEZ1 131.57 132.00 130.55 130.62 23398.0
2021-11-05  Live Cattle Future (front month) (December)   LEZ1 130.85 131.93 130.68 131.80 31050.0
2021-11-08  Live Cattle Future (front month) (December)   LEZ1 131.70 132.50 131.70 132.10 30902.0
2021-11-09  Live Cattle Future (front month) (December)   LEZ1 131.88 132.38 131.40 132.20 30503.0
2021-11-10  Live Cattle Future (front month) (December)   LEZ1 131.90 132.38 131.30 132.00 31903.0
2021-11

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7823 > 512). Running this sequence through the model will result in indexing errors


count     3143.000000
mean     12118.510022
std      11707.769667
min       2793.000000
50%       7435.000000
75%      12133.500000
90%      35953.000000
95%      36746.000000
99%      37755.660000
max      38392.000000
Name: input_length, dtype: float64
count    3143.000000
mean      131.379574
std        86.968900
min         6.000000
50%       113.000000
75%       169.500000
90%       257.000000
95%       320.900000
99%       393.000000
max       560.000000
Name: target_length, dtype: float64
Input > 512: 1.0
Target > 256: 0.10054088450524976
Raw table_data length (chars): 13166
      Date                                 Product Name Symbol   Open   High    Low  Close  Volume
2021-11-02  Live Cattle Future (front month) (December)   LEZ1 128.90 130.43 128.88 129.95 21275.0
2021-11-03  Live Cattle Future (front month) (December)   LEZ1 130.90 132.40 130.62 131.65 27572.0
2021-11-04  Live Cattle Future (front month) (December)   LEZ1 131.57 132.00 130.55 130.62 23398.0
2021-11-05  Liv

In [8]:
# %%capture
# %run -i "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/train_splitEDA.py"

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7823 > 512). Running this sequence through the model will result in indexing errors


In [9]:
# after = set(globals().keys())
# new_vars = after - before
# #variables from exported notebook above
# print(new_vars)

{'summarize_market_table', 'sample', 'parsed', 'token_length', 'test_file', 'linearize_table', 'before', 'test_df', 'build_encoder_input', 'sample_df', 'model_name', 'tokenizer', 'train_file', 'summary', '__file__', 'i', '_i9', 'train_df', 'save_dir', 'linearize_table_compact', 'StringIO', '_i8'}


### Build Encoder input and decoder targets
- Encoder processes the structured financial data withfull context, no generation order needed.
- Decoder generates text token by token
- Cross-attention lets the decoder ground each generated word back in the specific numbers/facts from the encoder. This will be helpful to check the factual consistency of hypothesis.
- Bidirectional encoder:The encoder will use full, non-causal self-attention. It needs the full self attention to see the entire table at once to build a complete representation.
- So, no masking the tokens on the input side.
- Decoder: The decoder will use causal masked self-attention since it generates autoregressively and can't peek at future output tokens.
- Models to use : **T5/ BART**

In [10]:
def build_encoder_input(row):
    return (
        "Generate a financial market report.\n\n"
        f"Instruction:\n"
        f"{row['instruction']}\n\n"
        f"Market summary:\n"
        f"{row['market_summary']}"
    )

In [11]:
for df in (train_df, test_df):
    df["market_summary"] = df["table_data"].apply(summarize_market_table)
    df["encoder_input"] = df.apply(build_encoder_input, axis=1)
    df["decoder_target"] = df["report"]

print(train_df[["encoder_input", "decoder_target"]].head(2))

                                       encoder_input  \
0  Generate a financial market report.\n\nInstruc...   
1  Generate a financial market report.\n\nInstruc...   

                                      decoder_target  
0  Cattle futures posted moderate to strong gains...  
1  Cattle prices are trying to turn higher, as th...  


In [12]:
# Inspect lengths before choosing truncation limits.
train_df["input_length"] = train_df["encoder_input"].apply(token_length)
train_df["target_length"] = train_df["decoder_target"].apply(token_length)

print("Input lengths")
display(train_df["input_length"].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nTarget lengths")
display(train_df["target_length"].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nTargets longer than 384 tokens:", (train_df["target_length"] > 384).sum())
print("Percentage longer than 384:", (train_df["target_length"] > 384).mean())

Input lengths


,input_length
count,3143.000000
mean,174.361120
std,6.731297
min,148.000000
50%,176.000000
75%,178.000000
90%,181.000000
95%,183.000000
99%,185.000000
max,188.000000



Target lengths


,target_length
count,3143.000000
mean,131.379574
std,86.968900
min,6.000000
50%,113.000000
75%,169.500000
90%,257.000000
95%,320.900000
99%,393.000000
max,560.000000



Targets longer than 384 tokens: 42
Percentage longer than 384: 0.013363028953229399


In [13]:
print("Targets > 128:",(train_df["target_length"] > 128).mean())
print("Targets > 256:",(train_df["target_length"] > 256).mean())

Targets > 128: 0.42666242443525293
Targets > 256: 0.10054088450524976


In [14]:
print("ENCODER INPUT\n")
print(train_df.iloc[0]["encoder_input"])

print("\n")
print("DECODER TARGET\n")
print(train_df.iloc[0]["decoder_target"])

ENCODER INPUT

Generate a financial market report.

Instruction:
Please act as an expert financial market analyst. Please generate a market report:
1. by analyzing the historical market data provided.
2. following the market report example provided.

Market summary:
Product: Live Cattle Future (front month) (December)
Symbol: LEZ1
Period start: 2021-11-02
Period end: 2021-12-02
Trading days: 132
Starting close: 129.95
Ending close: 170.90
Period high: 171.00
Period low: 128.88
Absolute price change: 40.95
Percentage price change: 31.51%
Daily return volatility: 1.25%
Maximum daily gain: 10.11%
Maximum daily loss: -4.69%
Average volume: 9696
Maximum volume: 31903
Minimum volume: 558
Overall price trend: Upward


DECODER TARGET

Cattle futures posted moderate to strong gains as cash trade stayed supportive in the live cattle market. Dec live cattle gained 1.650 to 137.650, and Feb cattle were .975 higher to 139.575. Feeders saw mixed, to mostly higher market as Jan feeders were slightly 

In [15]:
print("Targets > 384:",(train_df["target_length"] > 384).sum())
print("Percentage > 384:",(train_df["target_length"] > 384).mean())

Targets > 384: 42
Percentage > 384: 0.013363028953229399


### Shared configuration and Hugging Face datasets

### Model
1. Fine-tuned pretrained T5-small - Load T5 model : "t5-small"  - Chunk long sequences. Split long documents into 512‑token chunks and process them sequentially.
2. t5 base
2. t5_tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-large")
3. t5_model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-large")
4. LongT5 (4k–16k tokens) - not possible due to cost constraints

#### Convert Pandas DataFrames into Hugging Face datasets

wrap train_df/test_df in a HuggingFace Dataset, apply tokenize_batch, and fine-tune with Seq2SeqTrainer

Dataset + Seq2SeqTrainer training loop

In [20]:
MODEL_NAME = "google-t5/t5-small"
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 384
GENERATION_KWARGS = {
    "max_new_tokens": MAX_TARGET_LENGTH,
    "num_beams": 4,
    "early_stopping": True,
    }

In [ ]:
MODEL_DIRS = {
    "zero_shot": os.path.join(BASE_DIR, "models", "t5-small-zero-shot"),
    "finetuned": os.path.join(BASE_DIR, "models", "t5-small-finetuned"),
    }
for path in MODEL_DIRS.values():
    os.makedirs(path, exist_ok=True)

In [18]:
train_dataset = Dataset.from_pandas(
    train_df[["encoder_input", "decoder_target"]],
    preserve_index=False,)

test_dataset = Dataset.from_pandas(
    test_df[["encoder_input", "decoder_target"]],
    preserve_index=False,)

test_inputs = test_df["encoder_input"].tolist()
references = test_df["decoder_target"].tolist()

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 3143
})
Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 795
})


### Shared batched prediction function

In [19]:
def generate_predictions_batched(model,tokenizer,texts,batch_size=8,
                                 max_input_length=MAX_INPUT_LENGTH,generation_kwargs=None,):

    """Generate predictions in batches for zero shot or fine tuned seq2seq models"""

    if generation_kwargs is None:
        generation_kwargs = GENERATION_KWARGS

    model = model.to(DEVICE)
    model.eval()
    predictions = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

        with torch.no_grad():
            generated_ids = model.generate(**encoded, **generation_kwargs,)

        batch_predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        predictions.extend(batch_predictions)

    return predictions

### Shared eval metrics  function

In [23]:
rouge_metric = evaluate.load("rouge")
sacrebleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
bertscore_metric = evaluate.load("bertscore")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [24]:
def evaluate_predictions(predictions, references, model_label):
    """Return one flat dictionary suitable for a model-level results DataFrame."""
    rouge_scores = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True,)

    bleu_scores = sacrebleu_metric.compute(
        predictions=predictions,
        references=[[reference] for reference in references],)

    meteor_scores = meteor_metric.compute(
        predictions=predictions,
        references=references,)

    bert_scores = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="en",)

    return {
        "model": model_label,
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "sacrebleu": bleu_scores["score"],
        "meteor": meteor_scores["meteor"],
        "bertscore_precision": float(np.mean(bert_scores["precision"])),
        "bertscore_recall": float(np.mean(bert_scores["recall"])),
        "bertscore_f1": float(np.mean(bert_scores["f1"])),
    }

In [29]:
# Every model will append 1 metrics row and 1 prediction set
metrics_rows = []
prediction_frames = []

### T5 zero-shot baseline:
running the pretrained, non-fine-tuned T5-small on test set before comparing it to fine-tuned version.

1. Load pretrained tokenizer and model  
2. Generate batched predictions directly from raw test strings  
3. Evaluate predictions  
4. Append metrics and predictions to shared DataFrames  

A data collator is not required for zero-shot generation because there is no Trainer or training batch.

### Tokenize
Apply to both train and test datasets : We can see input_ids, attention_mask, labels

Run again when loading from checkpoint, because tokenized datasets are not saved automatically.

In [25]:
# load the zero-shot model
zero_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
zero_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

print("Loaded zero-shot model:", MODEL_NAME)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loaded zero-shot model: google-t5/t5-small


In [26]:
# Run this once to save the zero-shot model
zero_model.save_pretrained(MODEL_DIRS["zero_shot"])
zero_tokenizer.save_pretrained(MODEL_DIRS["zero_shot"])
print("Saved zero-shot checkpoint to:", MODEL_DIRS["zero_shot"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved zero-shot checkpoint to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/models/t5-small-zero-shot


### Generate zero-shot predictions

In [27]:
zero_predictions = generate_predictions_batched(
    model=zero_model,
    tokenizer=zero_tokenizer,
    texts=test_inputs,
    batch_size=8,
)

print("Generated predictions:", len(zero_predictions))
print("\nExample prediction:\n", zero_predictions[0])
print("\nReference:\n", references[0])

Generated predictions: 795

Example prediction:
 : 36.77 Percentage price change: 25.63% Daily return volatility: 1.37% Maximum daily gain: 13.01% Maximum daily loss: -2.85% Average volume: 8873 Maximum volume: 35921 Minimum volume: 510 Overall price trend: Upward

Reference:
 Oct live cattle gained 0.075 to 151.675, closing with a new contract high for the seventh straight day, and Dec slipped 0.825 to 153.300. Feeders were mostly lower with the exception of the front month with expiration on the 27th. Nov feeders traded 1.225 lower to 177.925. December cattle posted a new contract high early in the session, before turning lower on the day. This could be a sign of consolidation, or a loss of momentum after a strong rally. Cash trade was still undeveloped on Tuesday, but expectations are for cash trade to trend higher week over week. Southern asking prices are $152, with undefined bids. Beef retail values have jumped in the last two days. At midday Tuesday, Choice carcasses traded 3.95

## Evaluate and collect zero-shot results

In [30]:
zero_metrics = evaluate_predictions(
    predictions=zero_predictions,
    references=references,
    model_label="t5-small-zero-shot",
)
metrics_rows.append(zero_metrics)

zero_output_df = test_df.copy()
zero_output_df["model"] = "t5-small-zero-shot"
zero_output_df["prediction"] = zero_predictions
prediction_frames.append(
    zero_output_df[["model", "encoder_input", "decoder_target", "prediction"]]
)

display(pd.DataFrame([zero_metrics]))

,model,rouge1,rouge2,rougeL,sacrebleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,t5-small-zero-shot,0.057857,0.001057,0.040656,0.00402,0.026264,0.771497,0.780427,0.775841


In [31]:
# Free GPU memory before loading the training model.
del zero_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Fine-tuned T5-small

Order:

1. Load a fresh tokenizer and pretrained model  
2. Tokenize train and test datasets  
3. Create the dynamic data collator  
4. Define training arguments and Trainer  
5. Train and save the model  
6. Generate batched predictions using the saved/best model  
7. Evaluate and append results

WE're not using padding="max_length" to ensure that padded label tokens dont contribute to the loss. The data collator will dynamically pad each batch.

The tokenizer, data collator, and model are included in the same fine-tuning pipeline.

In [32]:
# Loading a fresh model and tokenizer for fine-tuning

ft_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ft_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Loaded fresh model for fine-tuning:", MODEL_NAME)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loaded fresh model for fine-tuning: google-t5/t5-small


### Tokenize train and test datasets

In [33]:
def tokenize_batch(examples):
    model_inputs = ft_tokenizer(
        examples["encoder_input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,)

    labels = ft_tokenizer(
        text_target=examples["decoder_target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,)

tokenized_test = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=test_dataset.column_names,)

print(tokenized_train)
print(tokenized_test)
print(tokenized_train[0].keys())

Map:   0%|          | 0/3143 [00:00<?, ? examples/s]

Map:   0%|          | 0/795 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3143
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 795
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


### Create the data collator
Instead of manually padding every example to max_input_length, data collater will handle the padding dynamically per batch

In [34]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=ft_tokenizer,
    model=ft_model,
    padding="longest",
    label_pad_token_id=-100,)

print(data_collator)

DataCollatorForSeq2Seq(tokenizer=T5Tokenizer(name_or_path='google-t5/t5-small', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, singl

### Training arguments and Trainer


In [ ]:
# # from transformers import Seq2SeqTrainingArguments
# #  loaded from check point -do not run again

# training_args = Seq2SeqTrainingArguments(
#     output_dir="./t5-small-financial-commentary",

#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=5e-5,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     weight_decay=0.01,
#     num_train_epochs=5,
#     predict_with_generate=True,
#     logging_steps=50,
#     save_total_limit=2,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
#     fp16=True, # for T4 GPU
#     report_to="none"
# )

In [35]:
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(BASE_DIR, "checkpoints", "t5-small-finetuning"),
    learning_rate=3e-4,
    lr_scheduler_type="linear",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

Create the trainer

In [36]:
trainer = Seq2SeqTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=ft_tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],)

print("Trainer created")

Trainer created


In [37]:
print(type(tokenized_train))
print(type(tokenized_test))

<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


### Model Training

In [38]:
#  run once, load from check point -do not run again
train_result = trainer.train()

Epoch,Training Loss,Validation Loss
1,3.530049,3.306724
2,3.289475,3.121701
3,3.221325,3.023994
4,3.031935,2.958968
5,3.010464,2.913040
6,2.884902,2.887745
7,2.847972,2.864227
8,2.808240,2.851084
9,2.836844,2.841419
10,2.731277,2.840806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [39]:
display(pd.DataFrame([train_result.metrics]))

,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss,epoch
0,727.5464,43.2,5.402,1.508984e+15,3.065431,10.0


### Save fine tuned model

In [40]:
trainer.save_model(MODEL_DIRS["finetuned"])
ft_tokenizer.save_pretrained(MODEL_DIRS["finetuned"])

with open(
    os.path.join(MODEL_DIRS["finetuned"], "training_metrics.json"),
    "w",
) as file:
    json.dump(train_result.metrics, file, indent=2)

print("Saved fine-tuned model to:", MODEL_DIRS["finetuned"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/models/t5-small-finetuned


In [ ]:
# load checkpoint in a later session

# ft_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIRS["finetuned"])
# ft_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIRS["finetuned"]).to(DEVICE)

# print("Loaded fine-tuned checkpoint.")

### Generate fine-tuned predictions in batches

In [41]:
fine_tuned_model = trainer.model.to(DEVICE)

fine_tuned_predictions = generate_predictions_batched(
    model=fine_tuned_model,
    tokenizer=ft_tokenizer,
    texts=test_inputs,
    batch_size=8,
)

print("Generated predictions:", len(fine_tuned_predictions))
print("\nExample prediction:\n", fine_tuned_predictions[0])
print("\nReference:\n", references[0])

Generated predictions: 795

Example prediction:
 Oct live cattle gained 0.025 to 142.750, and Oct cattle gained 0.025 to 143.750. Feeders saw mixed trade on Tuesday, as Oct feeders gained 0.025 to 143.750. Oct feeders traded 0.025 higher to 180.750. For the week, Oct feeders gained 0.025, and Oct added 0.025. For the week, Oct feeders were 0.025 higher. October feeders were 0.025 higher, and June added 0.025. October feeders are trading at a small discount to the index. The Feeder Cattle Cash Index was softer on the day, gaining 0.025 to 143.25. The cash market was supportive, but the market is still in an uptrend. The cattle market is still in an uptrend, but the market may be looking for direction.

Reference:
 Oct live cattle gained 0.075 to 151.675, closing with a new contract high for the seventh straight day, and Dec slipped 0.825 to 153.300. Feeders were mostly lower with the exception of the front month with expiration on the 27th. Nov feeders traded 1.225 lower to 177.925. Dec

### Evaluate and append fine_tuned_model results

In [42]:
fine_tuned_metrics = evaluate_predictions(
    predictions=fine_tuned_predictions,
    references=references,
    model_label="t5-small-finetuned",
)
metrics_rows.append(fine_tuned_metrics)

fine_tuned_output_df = test_df.copy()
fine_tuned_output_df["model"] = "t5-small-finetuned"
fine_tuned_output_df["prediction"] = fine_tuned_predictions
prediction_frames.append(
    fine_tuned_output_df[["model", "encoder_input", "decoder_target", "prediction"]]
)

display(pd.DataFrame([fine_tuned_metrics]))

,model,rouge1,rouge2,rougeL,sacrebleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,t5-small-finetuned,0.272943,0.096536,0.203133,6.403039,0.17776,0.876687,0.840908,0.858156


### Combine and save all experiment results

In [44]:
metrics_df = pd.DataFrame(metrics_rows)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

display(metrics_df)
display(predictions_df.head())

metrics_path = os.path.join(BASE_DIR, "model_metrics.csv")
predictions_path = os.path.join(BASE_DIR, "model_predictions.csv")

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

print("Saved model metrics to:", metrics_path)
print("Saved predictions to:", predictions_path)

,model,rouge1,rouge2,rougeL,sacrebleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,t5-small-zero-shot,0.057857,0.001057,0.040656,0.004020,0.026264,0.771497,0.780427,0.775841
1,t5-small-finetuned,0.272943,0.096536,0.203133,6.403039,0.177760,0.876687,0.840908,0.858156


,model,encoder_input,decoder_target,prediction
0,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,"Oct live cattle gained 0.075 to 151.675, closi...",: 36.77 Percentage price change: 25.63% Daily ...
1,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,Oct live cattle slipped .225 to 151.450 with e...,": 37.77 Percentage price change: 26,32% Daily ..."
2,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,"The cattle market came under selling pressure,...",: 142.72 Absolute price change: 36.88 Percenta...
3,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,"The cattle market was choppy on Friday, finish...",": 37.33 Percentage price change: 26,10% Daily ..."
4,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,October live cattle went off the board today a...,: 179.45 Period high: 182.38 Period low: 146.7...


## T5-base Model

In [45]:
# Free memory from T5-small before loading T5-base
del ft_model, trainer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### Model name and directory

In [46]:
MODEL_NAME = "google-t5/t5-base"

In [47]:
# Shared configs are same as t5-small
# MAX_INPUT_LENGTH = 256
# MAX_TARGET_LENGTH = 384
# GENERATION_KWARGS = {
#     "max_new_tokens": MAX_TARGET_LENGTH,
#     "num_beams": 4,
#     "early_stopping": True,
# }

MODEL_DIRS = {
    "zero_shot": os.path.join(BASE_DIR, "models", "t5-base-zero-shot"),
    "finetuned": os.path.join(BASE_DIR, "models", "t5-base-finetuned"),
}

1. Need to change batch_size to 4 instead of 8 for T4 GPU memory management. T5-base is 3 times more parameters than T5-small (220M vs. 60M), so it needs more VRAM per example during both training and generation.

2. Also will add gradient_accumulation_steps=2 in Seq2SeqTrainingArguments so the effective batch size, which drives the gradient update, stays at 8, and that will match the t5-small.

### Load T5-base model and zero shot it

In [48]:
zero_tokenizer_base = AutoTokenizer.from_pretrained("google-t5/t5-base")
zero_model_base = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-base").to(DEVICE)
print("Loaded zero-shot model:", MODEL_NAME)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded zero-shot model: google-t5/t5-base


In [49]:
# save the zero-t5-base model locally, run once.

zero_model_base.save_pretrained(MODEL_DIRS["zero_shot"])
zero_tokenizer_base.save_pretrained(MODEL_DIRS["zero_shot"])
print("Saved zero-shot base checkpoint to:", MODEL_DIRS["zero_shot"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved zero-shot base checkpoint to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/models/t5-base-zero-shot


### generate zero-shot base predictions

In [50]:
zero_pred_base = generate_predictions_batched(
    model=zero_model_base,
    tokenizer=zero_tokenizer_base,
    texts=test_inputs,
    batch_size=4,)

print("Generated predictions:", len(zero_pred_base))
print("\nExample prediction:\n", zero_pred_base[0])
print("\nReference:\n", references[0])

Generated predictions: 795

Example prediction:
 142.72 Period high: 182.38 Period low: 142.72 Period high: 182.38 Period low: 142.72 Absolute price change: 36.77 Percentage price change: 25.63% Daily return volatility: 1.37% Maximum daily gain: 13.01% Maximum daily loss: -2.85% Average volume: 8873 Maximum volume: 35921 Minimum volume: 510 Average volume: 8873 Maximum volume: 8873 Minimum volume: 35921 Minimum volume: 510 Average volume:

Reference:
 Oct live cattle gained 0.075 to 151.675, closing with a new contract high for the seventh straight day, and Dec slipped 0.825 to 153.300. Feeders were mostly lower with the exception of the front month with expiration on the 27th. Nov feeders traded 1.225 lower to 177.925. December cattle posted a new contract high early in the session, before turning lower on the day. This could be a sign of consolidation, or a loss of momentum after a strong rally. Cash trade was still undeveloped on Tuesday, but expectations are for cash trade to trend

### Evaluate zero_model_base and append results

In [52]:
zero_metrics_base = evaluate_predictions(
    predictions=zero_pred_base,
    references=references,
    model_label="t5-base-zero-shot",
)
metrics_rows.append(zero_metrics_base)

zero_output_df_base = test_df.copy()
zero_output_df_base["model"] = "t5-base-zero-shot"
zero_output_df_base["prediction"] = zero_pred_base
prediction_frames.append(
    zero_output_df_base[["model", "encoder_input", "decoder_target", "prediction"]]
)

display(pd.DataFrame([zero_metrics_base]))

,model,rouge1,rouge2,rougeL,sacrebleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,t5-base-zero-shot,0.057949,0.003745,0.040108,0.224147,0.032917,0.752887,0.77749,0.764631


## Fine-tuned T5-base

In [53]:
# free memory before fine-tuning
del zero_model_base
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [54]:
# load fresh model and tokenizer

ft_tokenizer_base = AutoTokenizer.from_pretrained(MODEL_NAME)
ft_model_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Loaded fresh model for fine-tuning:", MODEL_NAME)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loaded fresh model for fine-tuning: google-t5/t5-base


### Tokenize train and test datasets

In [55]:
def tokenize_batch(examples):
    model_inputs = ft_tokenizer_base(
        examples["encoder_input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,)

    labels = ft_tokenizer_base(
        text_target=examples["decoder_target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,)

tokenized_test = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=test_dataset.column_names,)

print(tokenized_train)
print(tokenized_test)
print(tokenized_train[0].keys())

Map:   0%|          | 0/3143 [00:00<?, ? examples/s]

Map:   0%|          | 0/795 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3143
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 795
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [56]:
# data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=ft_tokenizer_base,
    model=ft_model_base,
    padding="longest",
    label_pad_token_id=-100,
)

print(data_collator)

DataCollatorForSeq2Seq(tokenizer=T5Tokenizer(name_or_path='google-t5/t5-base', vocab_size=32100, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip

### Training args with smaller batch size

keeping all args same as t5-small, except batch_size and adding gradient_accumulation_steps=2, to keep the training set-up comparable between the two models.

Note: learning rate is same as t5-small for a controlled  comparision. However, T5-base can sometimes be less tolerant of high learning rates. After the first epoch if training and validation loss become unstable or produces nans or validation performance worsens, I'll rerun with lower learning rate 1e-4.



In [58]:
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(BASE_DIR, "checkpoints", "t5-base-finetuning"),
    learning_rate=3e-4,
    lr_scheduler_type="linear",
    num_train_epochs=10,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2, # effective batch size = 4 * 2 = 8, same as T5-small

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

### Create Trainer

In [59]:
trainer = Seq2SeqTrainer(
    model=ft_model_base,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=ft_tokenizer_base,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer created")

Trainer created


In [60]:
train_result_base = trainer.train()

Epoch,Training Loss,Validation Loss
1,5.914865,2.793881
2,5.442623,2.655070
3,5.221135,2.589106
4,4.877039,2.553757
5,4.637150,2.533776
6,4.319771,2.541224
7,4.219010,2.539181


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss,epoch
0,1659.1883,18.943,2.369,4.707951e+15,5.022631,7.0


In [61]:
display(pd.DataFrame([train_result_base.metrics]))

,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss,epoch
0,1659.1883,18.943,2.369,4.707951e+15,5.022631,7.0


In [64]:
#  save finetuned model

trainer.save_model(MODEL_DIRS["finetuned"])
ft_tokenizer_base.save_pretrained(MODEL_DIRS["finetuned"])

with open(
    os.path.join(MODEL_DIRS["finetuned"], "training_metrics.json"),
    "w",
) as file:
    json.dump(train_result_base.metrics, file, indent=2)

print("Saved fine-tuned model to:", MODEL_DIRS["finetuned"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/models/t5-base-finetuned


In [63]:
print(ft_model_base.config.model_type)
print(ft_model_base.config.d_model)

t5
768


In [ ]:
# load checkpoint in a later session
# ft_tokenizer_base = AutoTokenizer.from_pretrained(MODEL_DIRS["finetuned"])
# ft_model_base = AutoModelForSeq2SeqLM.from_pretrained(
#     MODEL_DIRS["finetuned"]
# ).to(DEVICE)
# print("Loaded fine-tuned checkpoint.")

### Generate fine-tuned predictions in batches

In [65]:
fine_tuned_model_base = trainer.model.to(DEVICE)

fine_tuned_pred_base = generate_predictions_batched(
    model=fine_tuned_model_base,
    tokenizer=ft_tokenizer_base,
    texts=test_inputs,
    batch_size=4, # lower batch size similar to training
)

print("Generated predictions:", len(fine_tuned_pred_base))
print("\nExample prediction:\n", fine_tuned_pred_base[0])
print("\nReference:\n", references[0])

Generated predictions: 795

Example prediction:
 Cattle futures saw mixed trade to end the week, as the cattle market saw selling pressure to start the week. Oct live cattle gained.075 to 142.750, and Dec live cattle gained.075 to 143.750. Feeders saw mixed trade as Oct feeders traded.075 lower to 179.375. October feeders traded.075 lower to 179.375. October feeders traded.075 lower to 179.375. October feeders traded.075 lower on the week. October live cattle futures are still trading at a discount to the cash market, which could be a limiting factor to the live cattle market. The cash market has been trending lower this week, but prices are still trading at a discount to the cash market. Retail values were firmer at midday (Choice: -1.09 to 256.29, Select: -1.29 to 249.29) with demand light at 61 midday loads. Feeder cattle saw strong buying strength as choice carcasses gained.27 to 264.27. Feeder cattle saw strong buying strength as choice carcasses gained.27 to 264.27. Feeder cattle

### Evaluate and append fine-tuned results

In [ ]:
# zero_metrics_base = evaluate_predictions(
#     predictions=zero_pred_base,
#     references=references,
#     model_label="t5-base-zero-shot",
# )
# metrics_rows.append(zero_metrics_base)

# zero_output_df_base = test_df.copy()
# zero_output_df_base["model"] = "t5-base-zero-shot"
# zero_output_df_base["prediction"] = zero_pred_base
# prediction_frames.append(
#     zero_output_df_base[["model", "encoder_input", "decoder_target", "prediction"]]
# )

# display(pd.DataFrame([zero_metrics_base]))

In [66]:
fine_tuned_metrics_base = evaluate_predictions(
    predictions=fine_tuned_pred_base,
    references=references,
    model_label="t5-base-finetuned",
)
metrics_rows.append(fine_tuned_metrics_base)

fine_tuned_output_df = test_df.copy()
fine_tuned_output_df["model"] = "t5-base-finetuned"
fine_tuned_output_df["prediction"] = fine_tuned_pred_base
prediction_frames.append(
    fine_tuned_output_df[["model", "encoder_input", "decoder_target", "prediction"]]
)

display(pd.DataFrame([fine_tuned_metrics]))

,model,rouge1,rouge2,rougeL,sacrebleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,t5-small-finetuned,0.272943,0.096536,0.203133,6.403039,0.17776,0.876687,0.840908,0.858156


In [67]:
#  combine and save al experiments results
metrics_df = pd.DataFrame(metrics_rows)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

display(metrics_df)
display(predictions_df.head())

,model,rouge1,rouge2,rougeL,sacrebleu,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,t5-small-zero-shot,0.057857,0.001057,0.040656,0.004020,0.026264,0.771497,0.780427,0.775841
1,t5-small-finetuned,0.272943,0.096536,0.203133,6.403039,0.177760,0.876687,0.840908,0.858156
2,t5-base-zero-shot,0.057949,0.003745,0.040108,0.224147,0.032917,0.752887,0.777490,0.764631
3,t5-base-zero-shot,0.057949,0.003745,0.040108,0.224147,0.032917,0.752887,0.777490,0.764631
4,t5-base-finetuned,0.286315,0.108871,0.213307,7.417643,0.194107,0.876012,0.846746,0.860887


,model,encoder_input,decoder_target,prediction
0,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,"Oct live cattle gained 0.075 to 151.675, closi...",: 36.77 Percentage price change: 25.63% Daily ...
1,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,Oct live cattle slipped .225 to 151.450 with e...,": 37.77 Percentage price change: 26,32% Daily ..."
2,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,"The cattle market came under selling pressure,...",: 142.72 Absolute price change: 36.88 Percenta...
3,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,"The cattle market was choppy on Friday, finish...",": 37.33 Percentage price change: 26,10% Daily ..."
4,t5-small-zero-shot,Generate a financial market report.\n\nInstruc...,October live cattle went off the board today a...,: 179.45 Period high: 182.38 Period low: 146.7...


In [68]:
metrics_path = os.path.join(BASE_DIR, "model_metrics.csv")
predictions_path = os.path.join(BASE_DIR, "model_predictions.csv")

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

print("Saved model metrics to:", metrics_path)
print("Saved predictions to:", predictions_path)

Saved model metrics to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/model_metrics.csv
Saved predictions to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/model_predictions.csv


### Evaluation
- Output commentary Vs. Reference commentary
- Output commentary vs. Underlying numerical facts

- Evaluation Metrics
  - Lexical
  - ROUGE
  - BLEU

- Semantic
  - BERTScore

- Factual grounding
  - LLM as a judge